In [1]:
from z3 import *
import numpy as np
import pandas as pd
from typing import List
import time
import pickle 
import pandas as pd
import ast
import bisect

### 1) Description
This notebook explains how SMT encodings of trained machine learning models can be used to compute abductive explanations for individual predictions. Specifically, we present an algorithm that takes as input an individual instance and a learned model encoded as an SMT/SAT formula, and computes an abductive explanation for the model's decision on that instance.

The implementation provided in this notebook follows Algorithm 1 presented in our paper, *On Explaining Proxy Discrimination and Unfairness in Individual Decisions Made by AI Systems* (AJCAI 2025). The algorithm identifies a minimal set of feature assignments that is sufficient to guarantee the prediction, thereby providing a formal and interpretable explanation of the decision.


In [2]:
def compute_explanation(s,individual,decision):
    '''This function is to computhe the explanations of the decision of a 
    certain individual based on its features'''
    features=[]
    remove=[]
    s.push()
    Not_d=0
    if decision == 0:
        Not_d=1

    for i in range(len(individual)):
        s.add(Real('i0'+str(i)) ==  individual[i])
    s.add(Int('decision' )== int(decision))
    resp= s.check()
    s.pop(1)
    if resp == sat:
         for feat in range(len(individual)):
                remove.append(feat)
                s.push()
                for i in range(len(individual)):
                    if i not in remove:
                        s.add(Real('i0'+str(i)) ==  individual[i])
                s.add(Int('decision' )== int(Not_d))
                rep= s.check()
                s.pop(1)
                if rep ==sat:
                    remove.pop()
                    features.append(feat)
    return features

In our implementation, features are represented by their indices rather than their names. Consequently, an individual instance is represented as a vector of feature values:

[
I = [v_1, v_2,..., v_n].
]

The corresponding feature set is therefore represented by the indices

[
F = {1, 2,..., n}.
]

The algorithm operates on these feature indices when searching for an abductive explanation. As a result, the explanation returned by the algorithm is a subset of feature indices that identifies the features responsible for the model's decision. Each index in the explanation points to a feature value in the individual instance whose presence is necessary to guarantee the prediction.


### 2) Implementation with the learned models

In this section, we load the trained models encoded as SMT files and define an example instance for which explanations will be computed. More specifically, we load the four saved models and compare the abductive explanations they generate for the same individual. This comparison illustrates how different models may rely on different subsets of features when producing the same prediction.


In [3]:
#1) Load a solver
def load_solver(path):
    s= Solver()
    s.from_file(path)
    return s

#2) Load a csv file
def profiles(path):
    data= pd.read_csv(path, sep=',')
    return(data)

def compute_decision(s, profil):
    prediction=0
    s.push()
    for i in range(len(profil)):
        s.add(Real('i0'+str(i)) ==  profil[i])
    s.add(Int('decision') == 0)
    resp= s.check()
    if resp== unsat:
        prediction=1
    s.pop()
    return prediction

In [4]:
LR_model= load_solver("LOGISTIC REGRESSION/model.smt2")
NN_model= load_solver("NEURAL NETWORK/model.smt2")
SGD_model= load_solver("SGD/model.smt2")

In [5]:
data= profiles("NEURAL NETWORK/binarized_GCD")

In [6]:
indidual_1= list(data.iloc[0, 1:-1]) #first element of the dataframe
indidual_1

[1, 1, 1, 1, 0, 0, 0, 0, 0, 1]

We first compute the predictions of the learned models for the selected individual and then generate the corresponding abductive explanations. The resulting explanations can be compared to understand how each model justifies its decision.

In [7]:
d_nn= compute_decision(NN_model,indidual_1)
d_nn

1

In [8]:
XP_1=compute_explanation(NN_model,indidual_1,d_nn)
XP_1

[4, 7, 9]

In [9]:
d_lr= compute_decision(LR_model,indidual_1)
d_lr

1

In [10]:
XP_2=compute_explanation(LR_model,indidual_1,d_lr)
XP_2

[7, 8, 9]

In [11]:
d_sgd= compute_decision(SGD_model,indidual_1)
d_sgd

1

In [12]:
XP_3=compute_explanation(SGD_model,indidual_1,d_sgd)
XP_3

[7]

##### Conclusions
As shown above, the explanations (XP_1), (XP_2), and (XP_3) obtained for the first individual from the neural network, logistic regression, and stochastic gradient descent models, respectively, are not identical. This highlights that different models may base their decisions on different combinations of features, even when producing the same prediction.

### 3) Compute an explanation under constraint of exclusion if exists

In some situations, it may be desirable to compute an explanation that satisfies additional constraints. In particular, one may wish to obtain an explanation that excludes a specific feature, provided that such an explanation exists. This can be useful when investigating the influence of sensitive attributes or suspected proxy variables on a model's decision.

In this section, we present an algorithm for computing abductive explanations that satisfy this exclusion criterion.

In [13]:
def compute_explanation_ind_constraint(s,individual,decision,constraint):
    '''This function is to computhe the explanations of the decision of a 
    certain individual based on its features and a specific constraint'''
    features=[]
    remove=[]
    remove.extend(constraint)
    s.push()
    Not_d= 0
    if decision==0:
        Not_d= 1
    for i in range(len(individual)):
        if i not in remove:
            s.add(Real('i0'+str(i)) ==  individual[i])
    s.add(Int('decision' )==Not_d)
    rep= s.check()
    s.pop(1)
    if rep ==sat:
        return None
    for feat in range(len(individual)):
        if feat in remove:
            continue
        remove.append(feat)
        s.push()
        for i in range(len(individual)):
            if i not in remove:
                s.add(Real('i0'+str(i)) ==  individual[i])
        s.add(Int('decision' )==Not_d)
        rep= s.check()
        s.pop(1)
        if rep ==sat:
            remove.pop()
            features.append(feat)
    return features

The fundamental difference between this algorithm and the one presented earlier is that the feature to be excluded is removed from the search space at the very beginning of the computation. Consequently, the algorithm searches only among candidate explanations that do not contain the specified feature.

To illustrate this approach, we use the same individual considered in the previous section. For each learned model, we attempt to compute an abductive explanation that excludes feature 7, since this feature appeared in all previously computed explanations. If such an explanation exists, the algorithm returns it; otherwise, it returns `*None*`, indicating that feature 7 is necessary to guarantee the prediction.


In [14]:
XP_1_prime= compute_explanation_ind_constraint(NN_model, indidual_1,d_nn,[7])
XP_1_prime

[2, 3, 4, 9]

In [15]:
XP_2_prime= compute_explanation_ind_constraint(LR_model, indidual_1,d_lr,[7])
XP_2_prime

[3, 4, 9]

In [16]:
XP_3_prime= compute_explanation_ind_constraint(SGD_model, indidual_1,d_sgd,[7])
XP_3_prime

[3, 4, 9]

## Conclusions
As shown above, each model was able to compute an alternative explanation that excludes feature 7. This demonstrates that a given prediction may admit multiple abductive explanations. In other words, there may exist several distinct subsets of features that are each sufficient to guarantee the same decision.

The existence of multiple abductive explanations highlights the fact that a model's prediction can often be justified in different ways, depending on which relevant features are considered.


### 4) Compute multiple explanations for an indiivudal

This section introduces an algorithm for enumerating all abductive explanations of an individual's decision. The approach is based on incremental hitting set computation, where each newly discovered explanation is used to generate blocking constraints. These constraints are added to the search space to prevent the computation of previously identified explanations and their supersets.

As a result, the algorithm systematically explores the explanation space and returns all subset-minimal abductive explanations associated with the decision.

In [17]:
def compute_incremental_hitting_set(E1,E2:List[int]):
    '''This function aims to compute the hitting set of two explanations'''
    hitting_set=[]
    combinations=[]
    for elem1 in E1:
        for elem2 in E2:
            if elem1 not in elem2:
                a= elem2.copy()
                a.append(elem1)
                combinations.append(a)
            else:
                hitting_set.append(elem2)
    terms= hitting_set.copy()
    for elem in combinations:
        rep= False
        i=0
        while rep==False and i < len(terms):
            if not set(terms[i]).issubset(elem):
                i+=1
            else:
                rep=True
        if rep==False:
            hitting_set.append(elem)
    return hitting_set

def compute_explanations_ind(s,individual,d):
    terms = []
    characters = []
    visited_constraint=[]
    to_be_visited_constraint=[]
    explanations= []
    constraints = [ [] ]
    while len(constraints)!=0:
        v_cons=constraints[0]
        axp= compute_explanation_ind_constraint(s, individual,d,v_cons)
        if axp != None:
            constraints= compute_incremental_hitting_set(axp,constraints)
            explanations.append(axp)
        else:
            constraints.remove(v_cons)
    return explanations

Again, we apply this algorithm to the same individual considered in the previous sections and evaluate it on each of the learned models. This allows us to enumerate all abductive explanations associated with the individual's prediction and to compare the explanation spaces across different models.

In [18]:
NN_xps= compute_explanations_ind(NN_model,indidual_1,d_nn)
NN_xps

[[4, 7, 9],
 [3, 7, 8, 9],
 [2, 3, 4, 9],
 [2, 3, 7, 9],
 [0, 3, 4, 6],
 [0, 3, 7, 9],
 [2, 3, 6, 7, 8],
 [0, 1, 3, 4],
 [2, 3, 4, 6, 8],
 [0, 3, 6, 7],
 [2, 3, 4, 7],
 [0, 4, 6, 7],
 [1, 2, 3, 4, 8],
 [0, 3, 4, 7]]

In [19]:
LR_xps= compute_explanations_ind(LR_model,indidual_1,d_lr)
LR_xps

[[7, 8, 9],
 [3, 4, 9],
 [4, 7],
 [1, 3, 8, 9],
 [0, 4, 8],
 [3, 7],
 [0, 1, 4, 9],
 [3, 4, 8],
 [0, 7],
 [0, 3, 9],
 [1, 7, 8],
 [0, 3, 8],
 [1, 3, 4],
 [0, 3, 4],
 [0, 1, 3]]

In [25]:
SGD_xps= compute_explanations_ind(SGD_model,indidual_1,d_sgd)
SGD_xps

[[7],
 [3, 4, 9],
 [1, 4, 9],
 [1, 3, 6, 9],
 [0, 4, 9],
 [0, 3, 9],
 [0, 1, 9],
 [1, 3, 4],
 [0, 4, 6],
 [0, 1, 3],
 [0, 3, 4],
 [0, 1, 4]]

###### Conclusions
Each list corresponds to the set of all subset-minimal abductive explanations for the individual's decision under a specific learned model. These explanations capture the alternative feature subsets that are sufficient to guarantee the model's prediction.

### 5) Compute an explanation under constraint of inclusion if exists

This section presents a method for computing an abductive explanation under an inclusion constraint. Specifically, the goal is to determine whether there exists an explanation that contains a given set of features.

The algorithm proposed here is non-recursive, as recursive search is not directly supported by Z3. Instead, it is based on the same enumeration procedure introduced for computing all abductive explanations of an individual's decision. The main difference is that, rather than storing every explanation found, the algorithm only retains explanations that satisfy the inclusion constraint.

If a valid explanation satisfying the constraint is found during the search, the algorithm terminates immediately and returns that explanation. Otherwise, in the worst case, it must enumerate all possible explanations before concluding that no explanation satisfying the constraint exists. Although this approach is not optimal from a computational perspective, it is a practical solution that can be readily implemented within the limitations of the Z3 framework.

In [28]:
def compute_explanation_include_ind(s,individual,d,constraint):
    visited_constraint=[]
    to_be_visited_constraint=[]
    explanations= []
    constraints = [ [] ]
    while len(constraints)!=0:
        v_cons=constraints[0]
        axp= compute_explanation_ind_constraint(s, individual,d,v_cons)
        if axp != None:
            constraints= compute_incremental_hitting_set(axp,constraints)
            if set(constraint).issubset(set(axp)):
                return axp
        else:
            constraints.remove(v_cons)
    return None

### 6) Let us refine the explanations using logic symbols (Optional)

In [29]:
def explanation(x, indices):
    
    '''This function turns indices into proper names of features'''
    
    features={'v0':0,'v1':0,'v2':0,'v3':0,'v4':0,'v5':0,
              'v6':0,'v7':0,'v8':0,'v9':0}
    keys= list(features.keys())
    explanation=[]
    individual=[] 
    for i in range(len(x)):
        if x[i]==0:
            individual.append('Not '+keys[i])
            if i in indices:
                explanation.append('Not '+keys[i])
        else:
            individual.append(keys[i])
            if i in indices:
                explanation.append(keys[i])
                    
    return explanation 

In [30]:
indidual_1

[1, 1, 1, 1, 0, 0, 0, 0, 0, 1]

In [31]:
explanation(indidual_1,XP_1)

['Not v4', 'Not v7', 'v9']

In [32]:
explanation(indidual_1,XP_2)

['Not v7', 'Not v8', 'v9']

In [33]:
explanation(indidual_1,XP_3)

['Not v7']